# DuckDB

This notebook demonstrates how to use [DuckDB](https://duckdb.org/) to search public SRA metadata directly from the free [SRA metadata S3 bucket](https://registry.opendata.aws/ncbi-sra/).

**What is DuckDB?**

DuckDB is a small, fast database engine that runs right inside your notebook. You do not need to set up a database server, create an AWS account, or download large metadata files before asking questions.

Instead, DuckDB lets you query files where they already live.

DuckDB is useful for exploring SRA metadata before downloading sequence data. It helps you filter by dates, organisms, sequencing strategies, library attributes, sample details, and other run metadata. In this notebook, we will use it as a lightweight search tool for finding SRA accessions that match a specific research question.

This tool can be used on Windows, Mac or Linux. It works with Python, R, or the command line.

**Example:** For this demo, we will look up the accessions of 2025 whole-genome sequencing experiments for the *Klebsiella* genus. The SRA metadata table can be filtered by date, by sequencing library attributes, and by sample/organism details.

---

## 1. Install DuckDB

Let's start by installing DuckDB into the Jupyter Enviornment. Outside of this exercise notebook, you will need to do this for your own machine using instruction present on the [DuckDB](https://duckdb.org/) website.

In [ ]:
# Install DuckDB into this Jupyter environment
%pip install duckdb

**NOTE:** After installing DuckDB using this command, you will need to restart the Jupyter kernel using `Menu Bar > Kernel > Restart Kernel`. You may need to re-open this notebook in order to get it working.

## 2. Connect to DuckDB and Run a First Query

Now we will import DuckDB, create a local in-process connection, and use SQL to scan SRA metadata files directly from the public S3 bucket.

This example searches for 2025 whole-genome sequencing runs from the *Klebsiella* genus and returns a small set of matching accessions. The `LIMIT 10` keeps the demo output manageable while you are learning the query pattern.

In [ ]:
import duckdb

# Create an in-process
con = duckdb.connect()

# Run your query
result = con.sql("""
SELECT acc, experiment, sample_acc, sra_study, organism, center_name, mbytes
FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*') -- parse parquet files stored on AWS ODP
WHERE organism ILIKE 'Klebsiella %' -- look for all organisms starting with Klebsiella
AND releasedate BETWEEN '2025-01-01' AND '2025-12-31'
AND librarysource='GENOMIC'
AND libraryselection='RANDOM'
AND assay_type = 'WGS'
LIMIT 10
""")
result

┌─────────────┬─────────────┬─────────────┬───────────┬───────────────────────┬────────────────────────────────────────────────────┬────────┐
│     acc     │ experiment  │ sample_acc  │ sra_study │       organism        │                    center_name                     │ mbytes │
│   varchar   │   varchar   │   varchar   │  varchar  │        varchar        │                      varchar                       │ int32  │
├─────────────┼─────────────┼─────────────┼───────────┼───────────────────────┼────────────────────────────────────────────────────┼────────┤
│ SRR36454719 │ SRX31478899 │ SRS27460597 │ SRP074197 │ Klebsiella pneumoniae │ AZDHS-API                                          │    476 │
│ SRR33365881 │ SRX28609830 │ SRS24881902 │ SRP582007 │ Klebsiella pneumoniae │ CHARITE - UNIVERSITAETSMEDIZIN BERLIN              │    521 │
│ ERR15831831 │ ERX15232258 │ ERS20912101 │ ERP151066 │ Klebsiella pneumoniae │ WELLCOME SANGER INSTITUTE                          │     69 │
│ DRR7

## 3. Refine the Results

The resulting python object can be filtered further with SQL by using it as a source table in an additional query.

The first query gives us a working set of matches. Because DuckDB returns a relation object, we can keep using SQL to filter those results further without rewriting the entire S3 query.

Here, we narrow the results to runs from one sequencing center.


In [2]:
result = con.sql("SELECT * FROM result WHERE center_name='WELLCOME SANGER INSTITUTE'")
result

┌─────────────┬─────────────┬─────────────┬───────────┬───────────────────────┬───────────────────────────┬────────┐
│     acc     │ experiment  │ sample_acc  │ sra_study │       organism        │        center_name        │ mbytes │
│   varchar   │   varchar   │   varchar   │  varchar  │        varchar        │          varchar          │ int32  │
├─────────────┼─────────────┼─────────────┼───────────┼───────────────────────┼───────────────────────────┼────────┤
│ ERR15831831 │ ERX15232258 │ ERS20912101 │ ERP151066 │ Klebsiella pneumoniae │ WELLCOME SANGER INSTITUTE │     69 │
└─────────────┴─────────────┴─────────────┴───────────┴───────────────────────┴───────────────────────────┴────────┘

## 4. See What Metadata Fields Are Available

Real-world metadata searches often start with a simple question: “What columns can I search?” DuckDB can describe the parquet files so we can inspect the available SRA metadata fields before building more detailed queries.


You can view all available attributes using `DESCRIBE FROM ...`


In [ ]:
result = con.sql("DESCRIBE FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*')").df()

result

## 5. Explore Values Before Filtering

Column names are only half of the story.

To write useful filters, it helps to see which values actually appear in a field. `SELECT DISTINCT` is a quick way to discover organisms, assay types, library strategies, center names, and other values that can be used in a query.




Filtering with SQL often requires knowing what attribute values are available. You can explore this using `SELECT DISTINCT`.



In [ ]:
result = con.execute("""
SELECT DISTINCT organism
FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*')
WHERE releasedate>'2026-05-01'
AND organism ILIKE '%metagenome%' --look for any string containing the word metagenome
""").df()
result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,null,key,default,extra
0,acc,VARCHAR,YES,None,None,None
1,assay_type,VARCHAR,YES,None,None,None
2,center_name,VARCHAR,YES,None,None,None
3,consent,VARCHAR,YES,None,None,None
4,experiment,VARCHAR,YES,None,None,None
5,sample_name,VARCHAR,YES,None,None,None
6,instrument,VARCHAR,YES,None,None,None
7,librarylayout,VARCHAR,YES,None,None,None
8,libraryselection,VARCHAR,YES,None,None,None
9,librarysource,VARCHAR,YES,None,None,None


## 6. Search Sample Attributes

Some SRA details are stored as sample-specific attributes rather than top-level columns. These attributes are kept as nested key/value pairs in the `attributes` column.

In this example, we look inside those nested attributes to find runs where the sample metadata mentions the `16S rRNA` sequencing method, then display the matching sample attributes next to the run metadata.





The SRA metadata also contains run-specific fields unique to the sample in question. For a given accession, these fields are stored together as a nested list of key/value pairs in the column `attributes`.  

Suppose you wanted to look for runs using the 16S rRNA sequencing method. You c could do this by iterating over key/value pairs in `attributes` with list_filter like so. The resulting sample metadata can be tabulated alongside run metadata

In [9]:
result = con.sql("""
-- first look for runs
WITH t AS (
  SELECT * FROM read_parquet('s3://sra-pub-metadata-us-east-1/sra/metadata/*')
  WHERE len(list_filter(attributes, x -> x.k = 'sequencing_method_sam' AND contains(x.v, '16S rRNA'))) > 0 -- check if list_filter is not empty
  LIMIT 10
  )
SELECT acc, organism, center_name, attr.k, attr.v
FROM t,UNNEST(attributes) AS t2(attr) -- cross join run metadata with sample metadata
""")
result

┌────────────┬──────────────┬──────────────────────────────────┬────────────────────────────┬──────────────────────────────────────────────────────────┐
│    acc     │   organism   │           center_name            │             k              │                            v                             │
│  varchar   │   varchar    │             varchar              │          varchar           │                         varchar                          │
├────────────┼──────────────┼──────────────────────────────────┼────────────────────────────┼──────────────────────────────────────────────────────────┤
│ ERR2588830 │ Homo sapiens │ VIRGINIA COMMONWEALTH UNIVERSITY │ bases                      │ 13768320                                                 │
│ ERR2588830 │ Homo sapiens │ VIRGINIA COMMONWEALTH UNIVERSITY │ bytes                      │ 8406939                                                  │
│ ERR2588830 │ Homo sapiens │ VIRGINIA COMMONWEALTH UNIVERSITY │ run_file_create_d

---

If you have any more specific questions, please visit the [DuckDB webpage](https://duckdb.org/) and the [DuckDB documentation repository](https://duckdb.org/docs/current/).

